In [2]:
pip install pandas numpy matplotlib seaborn scikit-learn pyodbc

Note: you may need to restart the kernel to use updated packages.


In [12]:
import pyodbc
import pandas as pd

conn = pyodbc.connect(
    "DRIVER={SQL Server};"
    "SERVER=DESKTOP-C876366;"   
    "DATABASE=SALESDW;"
    "Trusted_Connection=yes;"
)


In [18]:
query = "SELECT * FROM fact_sales"
df = pd.read_sql(query, conn)

df.head()

C:\Users\HP\AppData\Local\Temp\ipykernel_26320\1703505904.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,order_id,product_id,customer_id,order_date,region_id,sales,quantity,profit,margin,revenue,total_cost,growth,target,achievement,month_index
0,ca-2014-103800,off-pa-10000174,dp-13000,2014-01-03,1,16.45,2,5.55,0.3375,16.45,10.90,NaN,18.09,0.9091,1
1,ca-2014-112326,off-la-10003223,po-19195,2014-01-04,1,11.78,3,4.27,0.3625,11.78,7.51,-0.2836,12.96,0.9091,1
2,ca-2014-112326,off-st-10002743,po-19195,2014-01-04,1,272.74,3,-64.77,-0.2375,272.74,337.51,22.1446,300.01,0.9091,1
3,ca-2014-112326,off-bi-10004094,po-19195,2014-01-04,1,3.54,2,-5.49,-1.5500,3.54,9.03,-0.9870,3.89,0.9091,1
4,ca-2014-141817,off-ar-10003478,mb-18085,2014-01-05,2,19.54,3,4.88,0.2500,19.54,14.65,4.5186,21.49,0.9091,1


## Prévision des ventes

In [24]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Exemple : CA par mois
df['order_date'] = pd.to_datetime(df['order_date'])
df_grouped = df.groupby(df['order_date'].dt.to_period('M'))['revenue'].sum().reset_index()

df_grouped['order_date'] = df_grouped['order_date'].astype(str)

# Transformer en index numérique
df_grouped['t'] = np.arange(len(df_grouped))

X = df_grouped[['t']]
y = df_grouped['revenue']

model = LinearRegression()
model.fit(X, y)

# Prévision 3 mois
future = np.array([[len(df_grouped)], [len(df_grouped)+1], [len(df_grouped)+2]])
predictions = model.predict(future)

print(predictions)

[69957.52852837 70859.53587476 71761.54322116]


C:\Users\HP\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


## Segmentation clients (K-means)

In [27]:
from sklearn.cluster import KMeans

# Exemple features
data = df.groupby('customer_id').agg({
    'revenue': 'sum',
    'order_id': 'count'
}).reset_index()

X = data[['revenue', 'order_id']]

kmeans = KMeans(n_clusters=3, random_state=0)
data['Cluster'] = kmeans.fit_predict(X)

print(data.head())

  customer_id  revenue  order_id  Cluster
0    aa-10315  5563.56        11        1
1    aa-10375  1056.39        15        0
2    aa-10480  1790.51        12        0
3    aa-10645  5086.93        18        1
4    ab-10015   886.15         6        0


## Anomaly

In [43]:
mean = df_grouped['revenue'].mean()
std = df_grouped['revenue'].std()
df_grouped['Anomaly'] = df_grouped['revenue'].pct_change().apply(
    lambda x: 1 if x < -0.3 else 0
) 
print(df_grouped)

   order_date    revenue   t  Anomaly    Type
0     2014-01   14236.90   0        0  Actual
1     2014-02    4519.92   1        1  Actual
2     2014-03   55691.03   2        0  Actual
3     2014-04   28295.35   3        1  Actual
4     2014-05   23648.26   4        0  Actual
5     2014-06   34595.12   5        0  Actual
6     2014-07   33946.37   6        0  Actual
7     2014-08   27909.46   7        0  Actual
8     2014-09   81777.33   8        0  Actual
9     2014-10   31453.37   9        1  Actual
10    2014-11   78628.73  10        0  Actual
11    2014-12   69545.64  11        0  Actual
12    2015-01   18174.08  12        1  Actual
13    2015-02   11951.40  13        1  Actual
14    2015-03   38726.26  14        0  Actual
15    2015-04   34195.25  15        0  Actual
16    2015-05   30131.72  16        0  Actual
17    2015-06   24797.31  17        0  Actual
18    2015-07   28765.31  18        0  Actual
19    2015-08   36898.31  19        0  Actual
20    2015-09   64595.87  20      

## Affichage

In [46]:
df_grouped.to_csv("forecast1.csv", index=False)
data.to_csv("clusters.csv", index=False)